# Stage 10: WER + NE-F1 tables + acceptance gate  `[CPU]`
Paper Tables 3 & 4 — WER (syllable + word) and NE micro-F1, then the gate: ship the
adapter only if it matches/beats every baseline on val + hard. For `full`, repeat
predict/evaluate per seed and pass the reports to `evaluate.aggregate_reports` for
mean±std.

In [ ]:
# --- CarePath stage bootstrap (short by design) ---
import importlib.util, os, subprocess, sys
from pathlib import Path

def _find(start):
    for d in [start, *start.parents]:
        if (d / 'pyproject.toml').exists() and (d / 'apps' / 'api' / 'carepath').exists():
            return d
    return None

REPO = _find(Path.cwd().resolve())
if REPO is None and importlib.util.find_spec('google.colab'):
    url = os.environ.get('CAREPATH_REPO_URL', 'https://github.com/truong-tt/carepath.git')
    tok = os.environ.get('CAREPATH_GITHUB_TOKEN') or os.environ.get('GITHUB_TOKEN')
    if tok and url.startswith('https://github.com/'):
        url = url.replace('https://', f'https://x-access-token:{tok}@')
    subprocess.run(['git', 'clone', url, '/content/carepath'], check=True)
    REPO = Path('/content/carepath')
assert REPO, 'Open this notebook from inside the CarePath repo.'
os.chdir(REPO); sys.path.insert(0, str(REPO / 'apps' / 'api'))

PROFILE = 'smoke'   # <<< set to 'full' for the real ViMedCSS run
from carepath.gec.notebook import init_stage
CTX = init_stage(PROFILE); P = CTX.paths; PROF = CTX.profile


In [ ]:
# Install the GEC training stack (idempotent; needed once per Colab runtime).
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[training]'])


In [ ]:
CTX.run_step(['scripts/gec/evaluate.py', '--input', str(P.darag_preds), '--prediction-columns',
              'raw_asr', 'corrected_text', 'gec_pred', '--wer-output', str(P.darag_wer),
              '--ne-f1-output', str(P.darag_ne_f1)])
CTX.run_step(['scripts/gec/gate.py', '--report', str(P.darag_wer)])
import json
from carepath.gec.evaluate import render_ne_f1_table
print(render_ne_f1_table(json.load(open(P.darag_ne_f1, encoding='utf-8'))))
CTX.save([str(P.darag_wer), str(P.darag_ne_f1), str(P.leakage)])
